# NUTS-2 Geographical Hotspot Analysis

This notebook identifies geographical concentrations of waste hotspots - areas where multiple high-value NUTS-2 regions are spatially close to each other. This helps identify optimal locations for waste recovery infrastructure.

## Workflow
1. Load data and allocate waste to NUTS-2 regions
2. Cluster by Region × NACE × Waste profiles
3. Apply spatial clustering (DBSCAN) to find geographical concentrations
4. Identify cross-border opportunities

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.nuts2 import (
    # Data loading
    load_waste_generation,
    load_sbs_employment,
    load_nuts2_names,
    get_sbs_nuts2_employment,
    # Allocation
    allocate_waste_to_regions,
    add_economic_potential,
    # Clustering
    prepare_clustering_features,
    find_optimal_clusters,
    apply_clustering,
    apply_pca,
    get_cluster_profiles,
    # Geographical analysis
    load_nuts2_centroids,
    find_geographical_hotspots,
    get_hotspot_summary,
    get_cross_border_hotspots,
    calculate_cluster_density,
)

pd.options.display.max_columns = 50
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load and Prepare Data

In [ ]:
# Load waste generation data
wasgen = load_waste_generation()
print(f"Waste generation records: {len(wasgen):,}")
print(f"Countries: {wasgen['country_code'].nunique()}")
print(f"NACE activities: {wasgen['nace_r2'].nunique()}")
print(f"Waste types: {wasgen['waste'].nunique()}")

In [ ]:
# Load SBS employment data and NUTS2 names
sbs = load_sbs_employment(use_cache=True)
sbs_nuts2 = get_sbs_nuts2_employment(sbs)
nuts2_names = load_nuts2_names()

print(f"NUTS-2 employment records: {len(sbs_nuts2):,}")
print(f"Unique NUTS-2 regions: {sbs_nuts2['geo'].nunique()}")

## 2. Allocate Waste to NUTS-2 Regions

In [ ]:
# Allocate national waste to regions based on employment
regional_waste = allocate_waste_to_regions(wasgen, sbs_nuts2, nuts2_names)
regional_waste = add_economic_potential(regional_waste)

print(f"Allocated records: {len(regional_waste):,}")
print(f"Unique NUTS-2 regions: {regional_waste['nuts2_region'].nunique()}")
print(f"Total waste: {regional_waste['allocated_waste_tonnes'].sum()/1e9:.2f} billion tonnes")
print(f"Total economic potential: €{regional_waste['economic_potential_eur'].sum()/1e9:.1f} billion")

## 3. Cluster by Waste Profile

In [ ]:
# Prepare features and find optimal clusters
cluster_base, X_scaled, scaler = prepare_clustering_features(regional_waste)
print(f"Clustering {len(cluster_base):,} Region × NACE × Waste combinations")

best_k, silhouettes, inertias = find_optimal_clusters(X_scaled)
print(f"Optimal k by silhouette: {best_k}")

In [ ]:
# Apply clustering with 3 tiers (0=lowest, 2=highest value)
n_clusters = 3
clustered_data, kmeans = apply_clustering(cluster_base, X_scaled, n_clusters=n_clusters)

print(f"\nCluster distribution:")
for c in range(n_clusters):
    subset = clustered_data[clustered_data['cluster'] == c]
    print(f"  Cluster {c}: {len(subset):,} combos, "
          f"avg €{subset['economic_potential_eur'].mean()/1e6:.1f}M, "
          f"avg {subset['allocated_waste_tonnes'].mean()/1e3:.1f}k tonnes")

## 4. Geographical Hotspot Analysis

Now we identify spatial clusters where multiple high-value regions are geographically close. 

Uses **hierarchical clustering with complete linkage** to ensure all regions within a cluster are within the max diameter - this prevents the "chaining" problem where DBSCAN connects distant regions through intermediaries.

In [ ]:
# Load NUTS2 centroids
centroids = load_nuts2_centroids()
print(f"Loaded {len(centroids)} NUTS2 centroids")
centroids.head(10)

In [ ]:
# Find geographical hotspots among high-value cluster (cluster 2)
# max_diameter_km: maximum distance between ANY two regions in a cluster
# min_regions: minimum regions to form a cluster

min_cluster_threshold = 2  # Only the highest cluster (with 3 clusters total)

geo_hotspots = find_geographical_hotspots(
    clustered_data,
    min_cluster=min_cluster_threshold,
    max_diameter_km=400,   # All regions must be within 400km of each other
    min_regions=3,         # At least 3 regions per cluster
    aggregate_by='region',
    method='hierarchical'  # Prevents chaining problem
)

print(f"\nRegions in geographical clusters: {(geo_hotspots['geo_cluster'] >= 0).sum()}\"")
print(f"Isolated high-value regions: {(geo_hotspots['geo_cluster'] == -1).sum()}")

In [ ]:
# Summary of geographical hotspot clusters
hotspot_summary = get_hotspot_summary(geo_hotspots)
print("Geographical Hotspot Clusters (by economic potential):")
print("=" * 80)
for _, row in hotspot_summary.iterrows():
    print(f"\nGeo-Cluster {int(row['geo_cluster'])}: {int(row['n_regions'])} regions")
    print(f"  Countries: {row['countries']}")
    print(f"  Economic potential: €{row['total_economic_eur']/1e9:.2f}B")
    print(f"  Total waste: {row['total_waste_tonnes']/1e9:.2f}B tonnes")
    print(f"  Centroid: ({row['centroid_lat']:.2f}°N, {row['centroid_lon']:.2f}°E)")
    print(f"  Regions: {row['regions']}")

In [ ]:
# Calculate density of each geographical cluster
density = calculate_cluster_density(geo_hotspots)
print("\nGeographical clusters by economic density:")
density

## 5. Cross-Border Hotspots

Identify geographical clusters that span multiple countries - potential for cross-border waste recovery cooperation.

In [ ]:
# Find cross-border hotspots
cross_border = get_cross_border_hotspots(geo_hotspots, min_countries=2)

if len(cross_border) > 0:
    print(f"Cross-border hotspots: {cross_border['geo_cluster'].nunique()} clusters")
    print(f"Regions involved: {len(cross_border)}")
    
    # Summary by cluster
    for cluster_id in cross_border['geo_cluster'].unique():
        cluster_data = cross_border[cross_border['geo_cluster'] == cluster_id]
        countries = sorted(cluster_data['country_code'].unique())
        total_value = cluster_data['total_economic_eur'].sum()
        
        print(f"\nCross-border cluster {cluster_id}:")
        print(f"  Countries: {', '.join(countries)}")
        print(f"  Regions: {len(cluster_data)}")
        print(f"  Economic potential: €{total_value/1e9:.2f}B")
        
        for _, row in cluster_data.iterrows():
            print(f"    {row['nuts2_region']} ({row['country_code']}): {row['nuts2_name']} - €{row['total_economic_eur']/1e9:.2f}B")
else:
    print("No cross-border hotspots found with current parameters")

## 6. Visualize Geographical Hotspots

In [ ]:
# Map visualization of geographical hotspots with Europe basemap
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    use_cartopy = True
except ImportError:
    print("cartopy not installed, using plain plot. Install with: pip install cartopy")
    use_cartopy = False

if use_cartopy:
    # EPSG:3035 - ETRS89-LAEA Europe (official Eurostat projection)
    projection = ccrs.epsg(3035)
    
    fig, ax = plt.subplots(figsize=(10, 12), subplot_kw={'projection': projection})
    
    # Add map features
    ax.add_feature(cfeature.LAND, facecolor='#f5f5f5')
    ax.add_feature(cfeature.OCEAN, facecolor='#e6f3ff')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='#666666')
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='#999999', linestyle='-')
    ax.add_feature(cfeature.LAKES, facecolor='#e6f3ff', edgecolor='#999999', linewidth=0.3)
    
    # Set extent using EPSG:3035 coordinates (roughly covering EU)
    ax.set_extent([2500000, 6500000, 1500000, 5500000], crs=projection)
else:
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(-12, 32)
    ax.set_ylim(35, 70)

# Data coordinate reference system (lat/lon)
data_crs = ccrs.PlateCarree() if use_cartopy else None

# Plot isolated regions (noise) in gray
isolated = geo_hotspots[geo_hotspots['geo_cluster'] == -1]
scatter_kwargs = {'transform': data_crs, 'zorder': 5} if use_cartopy else {'zorder': 5}
ax.scatter(isolated['lon'], isolated['lat'], 
           c='lightgray', s=50, alpha=0.5, label='Isolated regions', **scatter_kwargs)

# Plot clustered regions with distinct colors
clustered = geo_hotspots[geo_hotspots['geo_cluster'] >= 0]

# Use a qualitative colormap with distinct colors
distinct_colors = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe',
    '#008080', '#e6beff', '#9a6324', '#800000', '#aaffc3',
    '#808000', '#ffd8b1', '#000075', '#808080', '#000000'
]

for i, cluster_id in enumerate(sorted(clustered['geo_cluster'].unique())):
    cluster_data = clustered[clustered['geo_cluster'] == cluster_id]
    
    # Get countries for this cluster
    countries = ', '.join(sorted(cluster_data['country_code'].unique()))
    
    # Size by economic potential
    sizes = (cluster_data['total_economic_eur'] / clustered['total_economic_eur'].max() * 300) + 50
    
    color = distinct_colors[i % len(distinct_colors)]
    scatter_kwargs = {'transform': data_crs, 'zorder': 10} if use_cartopy else {'zorder': 10}
    ax.scatter(cluster_data['lon'], cluster_data['lat'],
               c=color, s=sizes, alpha=0.7,
               label=f'Cluster {cluster_id}: {countries} ({len(cluster_data)} regions)',
               edgecolors='black', linewidths=0.5, **scatter_kwargs)
    
    # Label regions in each cluster
    for _, row in cluster_data.iterrows():
        if use_cartopy:
            ax.text(row['lon'], row['lat'], row['nuts2_region'],
                    fontsize=7, ha='center', va='bottom', alpha=0.8,
                    transform=data_crs, zorder=11)
        else:
            ax.annotate(row['nuts2_region'], (row['lon'], row['lat']),
                        fontsize=7, ha='center', va='bottom', alpha=0.8)

ax.set_title('Geographical Hotspots: High-Value NUTS-2 Region Clusters\n(size = economic potential)', 
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, framealpha=0.95)

if not use_cartopy:
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nuts2_geographical_hotspots.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Detailed Analysis by Geographical Cluster

In [ ]:
# For each geographical cluster, show the waste types and NACE activities
# that make these regions high-value

# First, link back to the detailed clustered data
high_value_detail = clustered_data[clustered_data['cluster'] >= min_cluster_threshold].copy()

# Add geo_cluster assignment
geo_mapping = geo_hotspots[['nuts2_region', 'geo_cluster']].drop_duplicates()
high_value_detail = high_value_detail.merge(geo_mapping, on='nuts2_region', how='left')

print("Waste profile breakdown by geographical cluster:")
print("=" * 80)

for geo_cluster in sorted(geo_hotspots[geo_hotspots['geo_cluster'] >= 0]['geo_cluster'].unique()):
    cluster_detail = high_value_detail[high_value_detail['geo_cluster'] == geo_cluster]
    cluster_regions = geo_hotspots[geo_hotspots['geo_cluster'] == geo_cluster]
    
    print(f"\n--- Geo-Cluster {geo_cluster} ---")
    print(f"Regions: {', '.join(sorted(cluster_regions['nuts2_region'].unique()))}\"")
    print(f"Countries: {', '.join(sorted(cluster_regions['country_code'].unique()))}\"")
    
    # Top waste types
    top_waste = cluster_detail.groupby(['waste', 'waste_description'])['economic_potential_eur'].sum().nlargest(5)
    print("\nTop waste types:")
    for (waste, desc), value in top_waste.items():
        print(f"  {waste}: {value/1e9:.2f}B - {desc[:50]}")
    
    # Top NACE activities
    top_nace = cluster_detail.groupby(['nace_r2', 'nace_activity'])['economic_potential_eur'].sum().nlargest(5)
    print("\nTop NACE activities:")
    for (nace, activity), value in top_nace.items():
        print(f"  {nace}: {value/1e9:.2f}B - {activity[:50]}")

## 8. Export Results

In [ ]:
# Export geographical hotspot data
geo_hotspots.to_csv('../data/processed/nuts2_geographical_hotspots.csv', index=False)
print(f"Saved: data/processed/nuts2_geographical_hotspots.csv ({len(geo_hotspots)} records)")

# Export summary
hotspot_summary.to_csv('../data/processed/nuts2_geo_hotspot_summary.csv', index=False)
print(f"Saved: data/processed/nuts2_geo_hotspot_summary.csv")

# Export detailed data with geo-cluster assignment
high_value_detail.to_csv('../data/processed/nuts2_high_value_with_geo.csv', index=False)
print(f"Saved: data/processed/nuts2_high_value_with_geo.csv ({len(high_value_detail)} records)")

In [ ]:
# Summary
print("\n" + "=" * 70)
print("GEOGRAPHICAL HOTSPOT ANALYSIS SUMMARY")
print("=" * 70)
print(f"Total high-value regions analyzed: {len(geo_hotspots)}")
print(f"Geographical clusters found: {(geo_hotspots['geo_cluster'] >= 0).nunique() - 1}")
print(f"Regions in clusters: {(geo_hotspots['geo_cluster'] >= 0).sum()}")
print(f"Isolated regions: {(geo_hotspots['geo_cluster'] == -1).sum()}")

if len(cross_border) > 0:
    print(f"\nCross-border clusters: {cross_border['geo_cluster'].nunique()}")
    print(f"Cross-border regions: {len(cross_border)}")

print(f"\nTotal economic potential in clusters: €{geo_hotspots[geo_hotspots['geo_cluster'] >= 0]['total_economic_eur'].sum()/1e9:.1f}B")